In [ ]:
PARQUET_PATH = (
    r'C:\Users\chrnolte\Dropbox\Data\earth\data\core\US\NC\_all\building'
    + r'\cheer\2026\US-NC_building-cheer-2026.parquet'
)
PARQUET_PATH = r'C:\Users\chrnolte\Dropbox\Data\earth\data\share\US\NC\_all\building\cheer\2026\US-NC_building-cheer-2026.parquet'
# PARQUET_PATH = (
# r'C:\Users\chrnolte\Dropbox\Data\earth\data\core\US\NC\BS\_all\building'
# + r'\cheer\2026\US-NC-BS_building-cheer-2026.parquet'
# )

In [ ]:
import geopandas as gpd

buildings = gpd.read_parquet(PARQUET_PATH)

In [ ]:
# from pathlib import Path
# from openplaces.io import to_parquet
# path = Path(PARQUET_PATH)
# to_parquet(buildings, path.with_name(f'{path.stem}_v2{path.suffix}'))

In [ ]:
buildings.sample(5).T

In [ ]:
buildings['openplaces_group_parcel_nsi'].value_counts().head(20)

In [ ]:
import pandas as pd

pd.concat(
    [
        buildings['source'].value_counts().apply('{0:,d}'.format),
        buildings['source'].value_counts(normalize=True).apply('{0:.1%}'.format),
    ],
    axis=1,
)

In [ ]:
mask_discrepancies = (
    buildings['openplaces_group_parcel'].notnull()
    & buildings['openplaces_group_nsi'].notnull()
    & buildings['openplaces_group_parcel'].ne(buildings['openplaces_group_nsi'])
)
mask_discrepancies.mean()

In [ ]:
buildings[mask_discrepancies]['openplaces_group_parcel_nsi'].value_counts().head(15)

In [ ]:
import numpy as np

ax = buildings['improvement_value_parcel'].hist(bins=np.logspace(2, 7.5, 50))
ax.set_xscale('log')

In [ ]:
ax = buildings['structure_value_nsi'].hist(bins=np.logspace(1, 7.5, 50))
ax.set_xscale('log')

In [ ]:
buildings.sample(5).T

In [ ]:
mask_year = buildings['year_built_parcel'].gt(1800) & buildings[
    'year_built_block_median_nsi'
].gt(1800)
buildings[mask_year].plot('year_built_parcel', 'year_built_block_median_nsi')

In [ ]:
buildings['year_built_parcel'].quantile(np.linspace(0, 1, 20))

In [ ]:
import matplotlib.pyplot as plt

XMIN, XMAX = 1930, 2020

fig, ax = plt.subplots(figsize=(5, 4))
quadmesh = ax.hist2d(
    buildings[mask_year]['year_built_parcel'],
    buildings[mask_year]['year_built_block_median_nsi'],
    cmap='Blues',
    bins=np.linspace(XMIN, XMAX, 90),
)
fig.colorbar(quadmesh[3], ax=ax, label='count')

ax.plot([XMIN, XMAX], [XMIN, XMAX], ls=':')
ax.set_ylabel('NSI (block group median)', size=12)
ax.set_xlabel('Parcel', size=12)
ax.set_title('Year built', size=15)